In [ ]:
"""
Raster processing: clip, reclassify, region group, and moving window analysis
Handles large rasters with parallel processing using open-source libraries
"""
import os
import re
import time
from pathlib import Path
from functools import partial
from multiprocessing import Pool, cpu_count

import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.features import shapes
from scipy import ndimage
from scipy.ndimage import generic_filter
import dask.array as da
from dask.diagnostics import ProgressBar

In [ ]:
# ============================================================================
# CONFIGURATION - Set these variables before running
# ============================================================================

# Clip parameters (optional)
CLIP_IN = r"S:\Mikayla\DATA\Projects\AF\NEW_WORKING\clip_in"
CLIP_MASK = r"S:\Mikayla\DATA\Projects\AF\NEW_WORKING\binary_mask\binary_mask_shrink40.tif"
CLIP_OUT = r"S:\Mikayla\DATA\Projects\AF\NEW_WORKING\clip_out"

# Reclassification parameters
RC_IN = r"S:\Mikayla\DATA\Projects\AF\Typology_collection9\2_MSPA\MSPA_renamed"
EDGE_RC_OUT = r"E:\typology\data\3_MovingWindow\MSPA_rc_edge"
AREA_RC_OUT = r"E:\typology\data\3_MovingWindow\MSPA_rc_area"
RC_TYPE = "edge"  # "edge" or "area"

# Region group parameters
RG_OUT = r"D:\NEW_WORKING\rg"
NEIGHBOR = "EIGHT"  # "FOUR" or "EIGHT"

# Reclass region group parameters
RC_RG_IN = r"D:\Mikayla_RA\RA_S25\NEW_WORKING\rg"
RC_RG_OUT = r"D:\Mikayla_RA\RA_S25\NEW_WORKING\rg_rc"

# Moving window parameters
EDGE_MW_IN = r"E:\typology\data\3_MovingWindow\MSPA_rc_edge"
AREA_MW_IN = r"E:\typology\data\3_MovingWindow\MSPA_rc_area"
PN_MW_IN = r"D:\NEW_WORKING\rg_rc"
MW_OUT = r"E:\typology\data\3_MovingWindow\mw_results"
MW_TYPE = "edge"  # "edge", "area", or "pn"
MW_RADIUS = 1000  # in map units (meters)
STAT = "SUM"  # "SUM" or "VARIETY"

# Processing parameters
N_WORKERS = cpu_count()
CHUNK_SIZE = 2048

In [ ]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def get_year(filename):
    """Extract 4-digit year from filename"""
    match = re.search(r"(\d{4})", filename)
    return match.group(1) if match else ""


def get_raster_files(directory, pattern="*.tif"):
    """Get list of raster files in directory"""
    rasters = sorted(Path(directory).glob(pattern))
    
    if not rasters:
        print(f"No rasters found in directory: {directory}")
        return []
    
    print(f"Found {len(rasters)} rasters")
    return rasters


# ============================================================================
# PROCESSING FUNCTIONS
# ============================================================================

def reclassify_mspa(input_path, rc_type, edge_out_dir, area_out_dir):
    """
    Reclassify MSPA output: edge (3, 103, 105) or area (3, 103, 105, 17, 117) to 1, rest to 0
    
    Args:
        input_path: Path to input raster
        rc_type: "edge" or "area"
        edge_out_dir: Output directory for edge
        area_out_dir: Output directory for area
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        year = get_year(input_path.name)
        
        if rc_type == "edge":
            output_dir = Path(edge_out_dir)
            output_path = output_dir / f"{year}_rc_edge.tif"
            target_values = [3, 103, 105]
        elif rc_type == "area":
            output_dir = Path(area_out_dir)
            output_path = output_dir / f"{year}_rc_area.tif"
            target_values = [3, 103, 105, 17, 117]
        else:
            print(f"Error: Undefined rc_type '{rc_type}'. Must be 'edge' or 'area'.")
            return None
        
        output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"Reclassifying {input_path.name} ({rc_type})...")
        
        with rasterio.open(input_path) as src:
            data = da.from_array(src.read(1), chunks=(CHUNK_SIZE, CHUNK_SIZE))
            
            # Create mask for target values
            mask = da.zeros_like(data, dtype=bool)
            for val in target_values:
                mask = mask | (data == val)
            
            # Explicit: target values = 1, everything else = 0
            result = da.where(mask, 1, 0).astype(np.uint8)
            
            with ProgressBar():
                result_computed = result.compute()
            
            kwargs = src.meta.copy()
            kwargs.update({
                'dtype': 'uint8',
                'nodata': None,
                'compress': 'lzw'
            })
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                dst.write(result_computed, 1)
        
        print(f"Successfully reclassified: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error reclassifying {input_path}: {e}")
        return None


def region_group(input_path, output_dir, connectivity=8):
    """
    Apply region grouping (connected component labeling) to identify patches
    
    Args:
        input_path: Path to input raster
        output_dir: Output directory
        connectivity: 4 or 8 neighbor connectivity
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        year = get_year(input_path.name)
        output_path = output_dir / f"{year}_area_rg.tif"
        
        print(f"Region grouping {input_path.name}...")
        
        # Map connectivity
        struct = ndimage.generate_binary_structure(2, connectivity // 4)
        
        with rasterio.open(input_path) as src:
            data = src.read(1)
            
            # Label connected components (exclude 0)
            binary = data > 0
            labeled, num_features = ndimage.label(binary, structure=struct)
            
            kwargs = src.meta.copy()
            kwargs.update({
                'dtype': 'int32',
                'nodata': None,
                'compress': 'lzw'
            })
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                dst.write(labeled.astype(np.int32), 1)
        
        print(f"Successfully region grouped: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error region grouping {input_path}: {e}")
        return None


def reclassify_rg(input_path, output_dir):
    """
    Reclassify region group raster: set value 1 to 0, keep rest
    
    Args:
        input_path: Path to input raster
        output_dir: Output directory
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        output_path = output_dir / f"{input_path.stem}_rc.tif"
        
        print(f"Reclassifying RG {input_path.name}...")
        
        with rasterio.open(input_path) as src:
            data = src.read(1)
            result = np.where(data == 1, 0, data)
            
            kwargs = src.meta.copy()
            kwargs.update({'compress': 'lzw'})
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                dst.write(result, 1)
        
        print(f"Successfully reclassified RG: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error reclassifying RG {input_path}: {e}")
        return None


def moving_window(input_path, output_dir, mw_type, radius, stat):
    """
    Apply moving window analysis with specified radius and statistic
    
    Args:
        input_path: Path to input raster
        output_dir: Output directory
        mw_type: Type identifier ("edge", "area", "pn")
        radius: Radius in map units
        stat: "SUM" or "VARIETY"
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        year = get_year(input_path.name)
        output_path = output_dir / f"{year}_{mw_type}_1km.tif"
        
        print(f"Moving window on {input_path.name}...")
        
        with rasterio.open(input_path) as src:
            data = src.read(1)
            pixel_size = src.transform[0]
            
            # Calculate radius in pixels
            radius_pixels = int(radius / pixel_size)
            
            # Create circular kernel
            y, x = np.ogrid[-radius_pixels:radius_pixels+1, -radius_pixels:radius_pixels+1]
            kernel = x**2 + y**2 <= radius_pixels**2
            
            # Apply focal statistic
            if stat == "SUM":
                result = ndimage.generic_filter(data.astype(np.float32), np.sum, footprint=kernel, mode='constant', cval=0)
            elif stat == "VARIETY":
                def variety(values):
                    return len(np.unique(values[values > 0]))
                result = ndimage.generic_filter(data, variety, footprint=kernel, mode='constant', cval=0)
            else:
                print(f"Error: Unsupported statistic '{stat}'")
                return None
            
            kwargs = src.meta.copy()
            kwargs.update({
                'dtype': 'float32',
                'nodata': None,
                'compress': 'lzw'
            })
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                dst.write(result.astype(np.float32), 1)
        
        print(f"Successfully processed moving window: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error in moving window {input_path}: {e}")
        return None


# ============================================================================
# BATCH PROCESSING
# ============================================================================

def process_rasters_parallel(process_func, input_files, n_workers=N_WORKERS, **kwargs):
    """Process multiple rasters in parallel"""
    if not input_files:
        return []
    
    print(f"Processing with {n_workers} workers...")
    
    func = partial(process_func, **kwargs)
    
    with Pool(processes=n_workers) as pool:
        outputs = pool.map(func, input_files)
    
    success_count = sum(1 for p in outputs if p)
    print(f"Process complete: {success_count}/{len(input_files)} succeeded")
    
    return outputs

In [ ]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main processing workflow"""
    print("Starting Processing")
      
    # # Reclassify
    # print("\nStarting Reclassification")
    # rc_start = time.time()
    # input_files = get_raster_files(RC_IN)
    # if input_files:
    #     rc_results = process_rasters_parallel(
    #         reclassify_mspa,
    #         input_files,
    #         rc_type=RC_TYPE,
    #         edge_out_dir=EDGE_RC_OUT,
    #         area_out_dir=AREA_RC_OUT
    #     )
    # print(f"Reclassification completed in {time.time() - rc_start:.2f} seconds")
    
    # # Region Group
    # print("\nStarting RegionGroup")
    # rg_start = time.time()
    # input_files = get_raster_files(AREA_RC_OUT)
    # if input_files:
    #     rg_results = process_rasters_parallel(
    #         region_group,
    #         input_files,
    #         output_dir=RG_OUT,
    #         connectivity=8 if NEIGHBOR == "EIGHT" else 4
    #     )
    # print(f"RegionGroup completed in {time.time() - rg_start:.2f} seconds")
    
    # # Reclass Region Group
    # print("\nStarting Reclass Region Group")
    # rc_rg_start = time.time()
    # input_files = get_raster_files(RC_RG_IN)
    # if input_files:
    #     rc_rg_results = process_rasters_parallel(
    #         reclassify_rg,
    #         input_files,
    #         output_dir=RC_RG_OUT
    #     )
    # print(f"Reclass Region Group completed in {time.time() - rc_rg_start:.2f} seconds")
    
    # Moving Window
    print("\nStarting Moving Window")
    mw_start = time.time()
    input_files = get_raster_files(EDGE_MW_IN)  # Change based on MW_TYPE
    if input_files:
        mw_results = process_rasters_parallel(
            moving_window,
            input_files,
            output_dir=MW_OUT,
            mw_type=MW_TYPE,
            radius=MW_RADIUS,
            stat=STAT
        )
    print(f"Moving window completed in {time.time() - mw_start:.2f} seconds")
    
    print("\nProcessing complete!")


if __name__ == "__main__":
    main()
